# V5W_01 — Preprocessing XDF → epoche (5words, TUTTI i soggetti)

Applica il **pipeline della tesi** (identico a EEG_39: band-pass 1–100 IIR, notch 50, resample 512→256, avg ref, µV, epoca `[marker, +1.5s]`) agli **XDF grezzi** di tutti i 41 soggetti → `data/5words_subjects/P###_S###/{parola}_{img|read}_{k}.csv`.

**Perché XDF e non le .set:** solo 17/41 hanno epoche pronte (pipeline di Iacomi, +ASR). Per una validazione **confrontabile con la tesi** serve lo *stesso* preprocessing su *tutti* → si riparte dal raw. Le `.set` restano un cross-check.

**Env: `daniele_311`** (mne + pyxdf). Gira in locale (gli XDF sono su OneDrive del Mac).

## Config

In [ ]:
# CONFIG — pipeline XDF→epoche, IDENTICO alla tesi (EEG_39), per TUTTI i soggetti
import warnings; warnings.filterwarnings('ignore')
import time, logging, re
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd, mne, pyxdf
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('v5w01')

project_root = next((p for p in [Path.cwd()] + list(Path.cwd().parents)
                     if (p / '.git').exists()), Path.cwd())
XDF_ROOT = Path('/Users/danieleuras/Library/CloudStorage/OneDrive-PolitecnicodiMilano/File di Francesco Iacomi - 5words')
OUT_ROOT = project_root / 'data' / '5words_subjects'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# Preprocessing — stessi parametri della tesi (EEG_39)
L_FREQ, H_FREQ, NOTCH_FREQ = 1.0, 100.0, 50.0
SFREQ_OUT = 256
N_CHAN    = 61
EPOCH_DUR = 1.5
N_SAMP    = int(EPOCH_DUR * SFREQ_OUT)   # 384
WORDS     = ['acqua', 'aiuto', 'mangiare', 'no', 'si']
OVERWRITE = False

assert XDF_ROOT.exists(), f'sorgente non trovata: {XDF_ROOT}'
print(f'XDF_ROOT: {XDF_ROOT.name}\nOUT_ROOT: {OUT_ROOT}\nN_SAMP={N_SAMP}  parole={WORDS}')


## Funzioni

In [ ]:
# FUNZIONI
def xdf_to_outdir(xdf_path: Path) -> Path:
    """.../sub-P0023/.../ses-S001/.../*.xdf -> OUT_ROOT/P023_S001 (padding normalizzato)."""
    parts = xdf_path.parts
    subj  = next(p for p in parts if p.startswith('sub-P'))
    sess  = next(p for p in parts if p.startswith('ses-S'))
    sid   = int(subj.replace('sub-P', ''))
    ses   = int(re.sub(r'\D', '', sess.replace('ses-S', '')))
    return OUT_ROOT / f'P{sid:03d}_S{ses:03d}'

def process_session(xdf_path: Path, out_dir: Path) -> dict:
    st = {'n_img': 0, 'n_read': 0, 'n_err': 0, 'skipped': False}
    if not OVERWRITE and out_dir.exists() and any(out_dir.glob('*_img_*.csv')):
        st['skipped'] = True; return st
    try:
        streams, _ = pyxdf.load_xdf(str(xdf_path))
    except Exception as e:
        log.error(f'load fail {xdf_path.name}: {e}'); st['n_err'] += 1; return st
    eeg = next((s for s in streams if s['info']['type'][0].upper() == 'EEG'), None)
    mrk = next((s for s in streams if s['info']['type'][0] == 'Markers'), None)
    if eeg is None or mrk is None:
        log.error(f'stream mancante {xdf_path.name}'); st['n_err'] += 1; return st

    raw_data = np.array(eeg['time_series'], dtype=np.float32).T[:N_CHAN, :]   # (61, n) drop ultimo canale
    sfreq_in = float(eeg['info']['nominal_srate'][0])
    t0 = float(eeg['time_stamps'][0])
    try:
        ch_info  = eeg['info']['desc'][0]['channel']
        ch_names = [ch['name'][0] for ch in ch_info[:N_CHAN]]
    except Exception:
        ch_names = [f'EEG{i:03d}' for i in range(N_CHAN)]

    info = mne.create_info(ch_names=ch_names, sfreq=sfreq_in, ch_types='eeg')
    raw  = mne.io.RawArray(raw_data * 1e-6, info, verbose=False)
    raw.filter(L_FREQ, H_FREQ, method='iir', verbose=False)
    raw.notch_filter(NOTCH_FREQ, verbose=False)
    if sfreq_in != SFREQ_OUT:
        raw.resample(SFREQ_OUT, verbose=False)
    raw.set_eeg_reference('average', verbose=False)
    proc = raw.get_data() * 1e6      # (61, n_res) in uV
    n = proc.shape[1]

    out_dir.mkdir(parents=True, exist_ok=True)
    counter = defaultdict(int)
    def save(ts, word, cond):
        idx = int(round((ts - t0) * SFREQ_OUT))
        if idx < 0 or idx + N_SAMP > n:
            return False
        k = counter[(word, cond)]; counter[(word, cond)] += 1
        ep = proc[:, idx:idx + N_SAMP].astype(np.float32)
        pd.DataFrame(ep).to_csv(out_dir / f'{word}_{cond}_{k:02d}.csv',
                                header=False, index=False, float_format='%.6f')
        return True

    for ts, m in zip(mrk['time_stamps'], [v[0] for v in mrk['time_series']]):
        if '_img' in m:
            w = m.replace('_img', '').lower()
            if w in WORDS: st['n_img'] += int(save(ts, w, 'img'))
        elif '_read' in m:
            w = m.replace('_read', '').lower()
            if w in WORDS: st['n_read'] += int(save(ts, w, 'read'))
    return st

print('Funzioni OK')


## Test su un file

In [ ]:
# TEST su un singolo XDF
test_xdf = next(XDF_ROOT.rglob('*.xdf'))
out = xdf_to_outdir(test_xdf); out_test = out.parent / (out.name + '_test'); 
import shutil; shutil.rmtree(out_test, ignore_errors=True)
t0 = time.time(); st = process_session(test_xdf, out_test); print(f'{time.time()-t0:.1f}s  {st}')
for f in sorted(out_test.glob('*.csv'))[:6]:
    a = pd.read_csv(f, header=None).values
    print(f'  {f.name}: {a.shape}  range=[{a.min():.1f},{a.max():.1f}] uV')
shutil.rmtree(out_test, ignore_errors=True)


## Loop principale

In [ ]:
# LOOP PRINCIPALE — tutti i soggetti, tutte le sessioni
all_xdf = sorted(XDF_ROOT.rglob('*.xdf'))
print(f'File XDF trovati: {len(all_xdf)}  |  soggetti: {len({p for x in all_xdf for p in x.parts if p.startswith("sub-P")})}')

tot = defaultdict(int)
for xdf in tqdm(all_xdf, desc='sessioni'):
    st = process_session(xdf, xdf_to_outdir(xdf))
    for k in ('n_img', 'n_read', 'n_err'): tot[k] += st[k]
    tot['skipped'] += int(st['skipped'])
print('\nTotale:', dict(tot))


## Sanity check

In [ ]:
# SANITY
subdirs = sorted(OUT_ROOT.glob('P*_S*'))
print(f'Sessioni scritte: {len(subdirs)}  |  soggetti: {len({d.name.split("_")[0] for d in subdirs})}')
from collections import Counter
c_img = Counter(); 
for f in OUT_ROOT.glob('P*_S*/*_img_*.csv'): c_img[f.name.split('_img_')[0]] += 1
print('Trial img/parola:', dict(c_img), '| tot img:', sum(c_img.values()))
ex = next(iter(sorted(OUT_ROOT.glob('P*_S*/*_img_*.csv'))))
a = pd.read_csv(ex, header=None).values
print(f'Esempio {ex.relative_to(OUT_ROOT)}: {a.shape}, [{a.min():.1f},{a.max():.1f}] uV')
